# Importar datos

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

# Configuración de rutas
data_path = r"C:\Users\xXSrBiscuitXx\Documents\GitHub\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"

try:
    df = pd.read_parquet(data_path)
    print(f"✓ Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas")
except Exception as e:
    print(f"✗ Error cargando el archivo: {e}")
    exit()

✓ Dataset cargado correctamente: 806 filas, 26 columnas


# Preparar variables


In [103]:
# Hit target average
pd.set_option("display.max.columns", None)
# Calculamos la media de hit target de cada trabajador y nos quedamos con un solo registro por trabajador 
df['Hit_target_avg'] = df.groupby('ID')['Hit_target'].transform('mean')

# Disciplinary failure
df["AtLeastOneDiscFailure"] = df.groupby("ID")["Disciplinary_failure"].transform(lambda x: int(x.max()))

# Suma de horas
df["TotalHours"] = df.groupby("ID")["Absenteeism_hours"].transform("sum")

# Conteo faltas injustificadas
df["UnjustifiedCount"] = df.groupby("ID")["Reason_absence"].transform(lambda x: (x == "Ausencia injustificada").sum())

In [104]:
df = df.drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)
df


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Hit_target,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,Reason_absence_numeric,Education_numeric,Month_absence_order,Day_week_order,Seasons_order,Hit_target_avg,AtLeastOneDiscFailure,TotalHours,UnjustifiedCount
0,6,Enfermedades del sistema musculoesquelético y ...,Abril,Jueves,Verano,189.0,29.0,13.0,33.0,246.288,91.0,0,High school,2,0,0,2,69.0,167.0,25.0,8.0,13,1,4.0,5,3,94.875000,0,72.0,0
1,16,Enfermedades del ojo y sus anexos,Junio,Miercoles,Verano,118.0,15.0,24.0,46.0,275.089,96.0,0,High school,2,1,1,0,75.0,175.0,25.0,8.0,7,1,6.0,4,3,97.500000,0,16.0,0
2,25,Consulta médica,Mayo,Jueves,Verano,235.0,16.0,8.0,32.0,237.656,99.0,0,Postgraduate,0,0,0,0,75.0,178.0,25.0,2.0,23,3,5.0,5,3,95.600000,0,42.0,0
3,12,"Lesiones, envenenamientos y consecuencias de o...",Julio,Viernes,Invierno,233.0,51.0,1.0,31.0,264.604,93.0,0,Graduate,1,1,0,8,68.0,178.0,21.0,2.0,19,2,7.0,6,1,96.142857,0,34.0,0
4,27,Consulta médica,Febrero,Viernes,Otono,184.0,42.0,7.0,27.0,302.585,99.0,0,High school,0,0,0,0,58.0,167.0,21.0,1.0,23,1,2.0,6,2,95.166667,0,25.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,132,"Lesiones, envenenamientos y consecuencias de o...",Febrero,Jueves,Otono,189.0,29.0,13.0,33.0,302.585,99.0,0,High school,2,0,0,2,69.0,167.0,25.0,8.0,19,1,2.0,5,2,99.000000,0,8.0,0
132,133,Consulta médica,Agosto,Viernes,Invierno,260.0,50.0,11.0,36.0,205.917,92.0,0,High school,4,1,0,0,65.0,168.0,23.0,4.0,23,1,8.0,6,1,92.000000,0,4.0,0
133,134,Consulta médica,Julio,Viernes,Invierno,179.0,51.0,18.0,38.0,239.554,97.0,0,High school,0,1,0,0,89.0,170.0,31.0,2.0,23,1,7.0,6,1,97.000000,0,2.0,0
134,135,Consulta médica,Marzo,Lunes,Otono,248.0,25.0,14.0,47.0,222.196,99.0,0,High school,2,0,0,1,86.0,165.0,32.0,2.0,23,1,3.0,2,2,99.000000,0,2.0,0


# Clustering

In [105]:
from sklearn.preprocessing import StandardScaler
from kmodes.kprototypes import KPrototypes

# Seleccionamos las columnas
dfClus = df[["Hit_target_avg", "AtLeastOneDiscFailure", "TotalHours", "UnjustifiedCount"]].copy()

# Escalamos solo las columnas numéricas
num_cols = ["Hit_target_avg", "TotalHours", "UnjustifiedCount"]
X_num = dfClus[num_cols].values
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

# Reconstruimos X agregando la columna categórica (falta disciplinaria)
X = np.hstack([X_num_scaled, dfClus[["AtLeastOneDiscFailure"]].values])

# Ahora la columna categórica es la última
cat_cols = [3]

# Entrenar K-Prototypes
kproto = KPrototypes(n_clusters=4, random_state=42, init='Huang')
clusters = kproto.fit_predict(X, categorical=cat_cols)

# Guardar clusters en el DataFrame
dfClus["cluster"] = clusters.astype(int)

resumen = dfClus.groupby("cluster").agg({
    "Hit_target_avg": "mean",
    "AtLeastOneDiscFailure": "mean",   # será la proporción de 1s
    "TotalHours": "mean",
    "UnjustifiedCount": "mean",
    "cluster": "count"                 # cantidad de empleados por cluster
}).rename(columns={"cluster": "n_empleados"})
print(resumen)

# Imprimir cost
print("Cost del clustering:", kproto.cost_)

# Ver la distribución de clusters
print(dfClus["cluster"].value_counts())

# Opcional: mostrar centroides
print("Centroides:")
print(kproto.cluster_centroids_)

         Hit_target_avg  AtLeastOneDiscFailure  TotalHours  UnjustifiedCount  \
cluster                                                                        
0             94.621384               0.615385  270.230769          1.000000   
1             93.381217               1.000000  286.000000          6.333333   
2             90.574775               0.135135    8.135135          0.054054   
3             96.480106               0.084337   14.265060          0.012048   

         n_empleados  
cluster               
0                 13  
1                  3  
2                 37  
3                 83  
Cost del clustering: 120.4814272332333
cluster
3    83
2    37
0    13
1     3
Name: count, dtype: int64
Centroides:
[[-1.80624556e-03  2.39934170e+00  6.97914028e-01  1.00000000e+00]
 [-3.69361969e-01  2.56589296e+00  5.70999628e+00  1.00000000e+00]
 [-1.20112372e+00 -3.68856558e-01 -1.91053265e-01  0.00000000e+00]
 [ 5.49074033e-01 -3.04113594e-01 -2.30528921e-01  0.00000000e+